[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/02_erstes_llm.ipynb)

# Sitzung 2 — Das Produkt & ein LLM zum Laufen bringen

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI · 14.10.2026**

Heute baut ihr den kleinsten Baustein des Review Radar selbst: einen **Klassifikator**, der eine Bewertung per LLM einordnet. Und ihr messt, wie verlässlich er ist.

**Ablauf:**

1. **Setup, Daten & Anatomie eines API-Aufrufs (Abschnitt 0.-2.):** Korpus erzeugen und mit pandas untersuchen; einen Aufruf zerlegen.
2. **Prompt als Funktionsdesign (Abschnitt 3.-4.):** `classify()` selbst schreiben, zero-shot vs. few-shot, erster Abgleich mit der Wahrheit.
3. **Nichtdeterminismus, quantifiziert (Abschnitt 5.-6.):** Softmax & Temperatur; dieselbe Frage 10× stellen und die Streuung messen.

## 0. Setup

Einmal pro Sitzung ausführen.

> 💡 **Was passiert hier?** Euer Code läuft nicht auf eurem Laptop, sondern auf einer virtuellen Maschine bei Google. pandas & Co. sind dort vorinstalliert; `!pip install` ergänzt den Rest. Die Maschine ist **temporär**: Nach Inaktivität (spätestens nach 12 h) wird sie gelöscht, samt installierten Paketen und Variablen. Das Notebook selbst bleibt erhalten.

In [1]:
!pip install anthropic --quiet

import json, re, os, time, math, random
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 90)
print("Setup fertig.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.8 MB/s eta 0:00:00
Setup fertig.


### API-Schlüssel einrichten (einmalig, 2 min)

Wir nutzen **Claude Haiku 4.5** von Anthropic. Den Kurs-Schlüssel findet ihr in **Moodle**.

1. In Colab links auf 🔑 **Secrets** klicken
2. **Neues Secret:** Name `ANTHROPIC_API_KEY`, Wert = der Kurs-Schlüssel
3. **„Notebook access"** einschalten

> ⚠️ Den Schlüssel **nie** in den Code, in Screenshots oder in Chats schreiben: Notebooks werden geteilt und landen auf GitHub. Secrets bleiben in eurem Google-Konto. Der Schlüssel hat ein Ausgabenlimit für den ganzen Kurs, geht also sparsam damit um.

Kein Schlüssel zur Hand? Dann wechselt das Notebook automatisch auf den **Mock**: eine Attrappe statt eines echten Modells. Alles läuft trotzdem.

In [2]:
BACKEND = "anthropic"                # "anthropic" | "mock"
ANTHROPIC_MODEL = "claude-haiku-4-5"

In [3]:
# LLM-Anbindung: einmal ausführen. Lesen lohnt sich, muss aber nicht.
def get_secret(name):
    """Liest ein Secret aus Colab, sonst aus Umgebungsvariablen."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

# ---- Mock: ein kleines, ehrliches Spielzeug-LLM --------------------------
# Es liest NUR die Bewertung (zwischen """ ... """), ignoriert eure Anweisungen
# und würfelt sein Label aus Wortsignalen. Sarkasmus versteht es nicht.
_POS = ["hervorragend", "satte", "top", "ganzen tag", "bequem", "sofort", "klasse",
        "übersichtlich", "perfekt", "klar", "begeistert", "gut", "empfehl", "zufrieden",
        "liebe", "gelungen", "gelohnt", "richtig", "super", "toll", "great", "amazing",
        "happy", "comfortable", "best"]
_NEG = ["stürzt", "lädt nicht", "rauscht", "teuer", "reagiert kaum", "billig", "leer",
        "bricht", "fallen", "matschig", "enttäusch", "schlecht", "finger weg", "frustriert",
        "schade", "ärgerlich", "reklamiert", "nie wieder", "zurückgeschickt", "vorsicht",
        "kaputt", "disaster", "disappointed", "weak", "cheap", "stopped", "tot"]
_NEU = ["okay", "zweck", "durchschnittlich", "nichts besonderes", "noch nicht viel"]
MOCK_LABELS = ["positive", "negative", "mixed", "neutral"]

def _extract_review(user_msg):
    m = re.findall(r'"""(.*?)"""', user_msg, re.DOTALL)
    return m[-1] if m else user_msg.strip().splitlines()[-1]

def mock_logits(review):
    t = review.lower()
    p = sum(w in t for w in _POS); n = sum(w in t for w in _NEG); u = sum(w in t for w in _NEU)
    return 2.0 * np.array([1.2*p - 0.8*n,            # positive
                           1.2*n - 0.8*p,            # negative
                           1.4*min(p, n) + 0.1,      # mixed
                           0.4 + 1.5*u - 0.4*(p+n)]) # neutral

def sample_from_logits(logits, temperature):
    if temperature == 0:
        return int(np.argmax(logits))
    z = logits / temperature
    probs = np.exp(z - z.max()); probs /= probs.sum()
    return int(np.random.choice(len(logits), p=probs))

def _mock_call(system, user, temperature, max_tokens):
    review = _extract_review(user)
    i = sample_from_logits(mock_logits(review), 1.0 if temperature is None else temperature)
    text = json.dumps({"sentiment": MOCK_LABELS[i], "reason": "Mock: Wortsignale gezählt."},
                      ensure_ascii=False)
    est = lambda s: max(1, len(s) // 4)   # grobe Schätzung: ~4 Zeichen pro Token
    return {"text": text, "input_tokens": est((system or "") + user), "output_tokens": est(text),
            "stop_reason": "end_turn", "model": "mock", "raw": None}

# ---- Echtes Backend: Anthropic -------------------------------------------
_client = None

def _anthropic_call(system, user, temperature, max_tokens):
    global _client
    import anthropic
    if _client is None:
        _client = anthropic.Anthropic(api_key=get_secret("ANTHROPIC_API_KEY"))
    kwargs = dict(model=ANTHROPIC_MODEL, max_tokens=max_tokens,
                  messages=[{"role": "user", "content": user}])
    if system:
        kwargs["system"] = system
    if temperature is not None:
        # Neuere SDK-Versionen kennen den Parameter nicht mehr: direkt in den Request-Body
        kwargs["extra_body"] = {"temperature": temperature}
    r = _client.messages.create(**kwargs)
    text = next((b.text for b in r.content if hasattr(b, "text")), "")
    return {"text": text, "input_tokens": r.usage.input_tokens,
            "output_tokens": r.usage.output_tokens, "stop_reason": r.stop_reason,
            "model": r.model, "raw": r}

def call_llm(user, system=None, temperature=None, max_tokens=300):
    """Ein Aufruf: Text rein, Text raus. temperature=None heißt: Standard des Anbieters."""
    fn = {"mock": _mock_call, "anthropic": _anthropic_call}[BACKEND]
    return fn(system, user, temperature, max_tokens)

# Automatischer Rückfall auf den Mock, wenn kein Schlüssel gefunden wird
if BACKEND == "anthropic" and not get_secret("ANTHROPIC_API_KEY"):
    print("⚠️ Kein Secret ANTHROPIC_API_KEY gefunden: Wechsel auf den Mock.")
    BACKEND = "mock"
print(f"Backend: {BACKEND}" + (f" ({ANTHROPIC_MODEL})" if BACKEND == "anthropic" else ""))

Backend: anthropic (claude-haiku-4-5)


## 1. Unser Datensatz: synthetische Bewertungen

Wir erzeugen den Korpus selbst: Bewertungen zum **fiktiven** Produkt „Nimbus Q2" der erfundenen Marke „Nimbus Audio".

### Warum nicht echte Daten?

Es gibt sie, und zwar in riesigem Umfang:
- **Amazon Reviews'23** (McAuley Lab, UC San Diego) enthält rund **571 Mio. Bewertungen** aus 1996–2023 in 33 Produktkategorien.
- Ihr könnt ihn im Browser ansehen: [Hugging Face: Amazon-Reviews-2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023).
- Auf der [Projektseite](https://amazon-reviews-2023.github.io/) seht ihr unter „Load User Reviews" einen Beispiel-Datensatz mit allen Feldern (`rating`, `title`, `text`, `verified_purchase` …) und die Download-Tabelle je Kategorie.

Aber: Die Bewertungen unterliegen Amazons Nutzungsbedingungen. Diese beschränken die Nutzung (v.a. auf Forschung) und verbieten die Weitergabe. Wir dürften ihn also weder in unser Repo legen noch frei für den Kurs verwenden (Sitzung 11)

### Wie entsteht unser Korpus?

- **Baukasten statt Freitext:** Jede Bewertung wird aus Bausteinen zusammengesetzt, also Einleitung + 1–2 Aspekte (Klang, Akku, App, Passform …). Dadurch sind die meisten Texte einzigartig.
- **Fester Mix:** 34 % positiv · 26 % negativ · 12 % gemischt · 8 % neutral · 8 % Sarkasmus · 5 % Fakes · 5 % Englisch · 2 % Müll, plus 3 absichtliche Duplikate.
- **Sterne passend zum Inhalt, mit Rauschen:** positiv meist 4–5, negativ meist 1–2, Fakes immer 5.
- **Zeitverlauf eingebaut:** Negative Bewertungen häufen sich später im Jahr. Ein Trend, den wir in Sitzung 7 wiederfinden.
- **Reproduzierbar:** fester Zufalls-Seed. Alle bekommen exakt denselben Datensatz.
- **Wahrheit inklusive:** Zu jeder Zeile kennen wir das „richtige" Label.

### Ist er repräsentativ? Nein, und das ist Absicht.

Echte Bewertungen folgen typischerweise einer **J-Verteilung**: sehr viele 5-Sterne, einige 1-Stern, wenig dazwischen¹. Unser Korpus ist dagegen **bewusst angereichert**: mehr Negatives, mehr Grenzfälle, mehr Müll als in der Realität. So kommen alle Fehlerfälle oft genug vor, um sie zu untersuchen.

> **Nicht:** „Synthetisch = realistisch."
> **Sondern:** Synthetisch = **kontrolliert**. Wir bauen das Chaos gezielt ein und wissen genau, wo es steckt.

---
<sub>¹ Hu, N., Zhang, J., Pavlou, P. A. (2009): „Overcoming the J-shaped Distribution of Product Reviews", *Communications of the ACM* 52(10).</sub>

In [4]:
# Generator: einmal ausführen. Lesen lohnt sich, muss aber nicht.
"""
Generator fuer synthetische Bewertungen (BDA-Kurs).

Fiktives Produkt: "Nimbus Q2" von der erfundenen Marke "Nimbus Audio".
Voellig fiktiv -> keine echte Marke/Person, keine Lizenzfrage.

Setzt Bewertungen aus variierten Vorlagen zusammen, damit die meisten
EINZIGARTIG sind (realistisch) - mit gezielt eingebautem Daten-Chaos:
  - gemischtes Sentiment, Sarkasmus, Fakes, etwas Englisch, Muell-Zeilen
  - ein paar ABSICHTLICHE exakte Duplikate (fuer die Dedup-Lektion)
  - Wahrheits-Marker (true_sentiment, is_sarcastic, is_fake) fuer spaetere Evaluation
"""
import random
from datetime import date, timedelta

SEED = 42
PRODUCT = "Nimbus Q2"
BRAND = "Nimbus Audio"

# Aspekt-Bausteine
POS = ["der Klang ist hervorragend", "satte Bässe", "die Geräuschunterdrückung ist top",
       "der Akku hält den ganzen Tag", "sitzt super bequem", "Bluetooth verbindet sofort",
       "top verarbeitet", "klasse für den Preis", "die App ist übersichtlich",
       "die Passform ist perfekt", "der Sound ist klar und ausgewogen"]
NEG = ["die App stürzt ständig ab", "der rechte Ohrhörer lädt nicht mehr",
       "die Geräuschunterdrückung rauscht", "viel zu teuer", "die Touch-Steuerung reagiert kaum",
       "das Case wirkt billig", "der Akku ist nach einer Stunde leer",
       "die Verbindung bricht ab", "sie fallen leicht aus dem Ohr", "der Bass ist matschig"]

POS_OPENERS = ["Bin begeistert:", "Wirklich gut:", "Kann ich empfehlen –", "Top Kauf.",
               "Sehr zufrieden:", "Absolute Kaufempfehlung.", "Ich liebe sie:",
               "Klare Sache:", "Rundum gelungen:", "Volle Punktzahl:", "Endlich zufrieden:",
               "Was soll ich sagen –", "Genau richtig:", "Bestellung hat sich gelohnt:"]
NEG_OPENERS = ["Enttäuschend:", "Leider schlecht:", "Finger weg –", "Bin frustriert:",
               "Nicht zu empfehlen.", "Schade um das Geld:", "Ärgerlich:",
               "Reklamiert:", "Bin raus:", "Nie wieder:", "Herbe Enttäuschung:",
               "Das war nichts:", "Zurückgeschickt:", "Vorsicht:"]
NEU_TEMPLATES = ["Ganz okay, {a}, aber nichts Besonderes.",
                 "Erfüllt seinen Zweck. {a_cap}.",
                 "Durchschnittlich. {a_cap}, mehr nicht.",
                 "Habe sie seit Kurzem, {a} – kann noch nicht viel sagen."]
SARCASTIC = ["Super, schon nach drei Tagen kaputt. Echt klasse Qualität.",
             "Toll, dass die App jedes Mal abstürzt. Genau das wollte ich.",
             "Wunderbar, 200 Euro für Ohrhörer, die nach links driften. Ein Traum.",
             "Ganz großes Kino, der Akku hält sagenhafte 40 Minuten.",
             "Klasse, nach einer Woche nur noch Rauschen. Wirklich durchdacht.",
             "Perfekt, der linke fällt ständig raus. Genau mein Wunsch.",
             "Herrlich, die Verbindung bricht alle fünf Minuten ab. Danke auch.",
             "Sensationell, das Case bricht beim ersten Öffnen. Qualität eben.",
             "Bravo, nach dem Update ist die Hälfte der Funktionen weg.",
             "Fantastisch leise – weil nach zwei Tagen einfach tot."]
FAKE = ["BESTES PRODUKT EVER!!! Kauft bei www.super-deals-guenstig.example!!!",
        "5 Sterne 5 Sterne bester shop schnelle lieferung AAA+++",
        "Gutschein Code NIMBUS100 auf meiner Seite jetzt klicken!!!",
        "amazing product best quality buy now discount link in profile",
        "TOP TOP TOP unbedingt kaufen billigster preis hier klicken",
        "gratis versand nur heute!!! rabattcode DEAL22 einlösen!!!",
        "beste kopfhoerer der welt jetzt zuschlagen link im profil",
        "WOW einfach WOW kaufen kaufen kaufen bester preis garantiert",
        "unglaublich guenstig hier klicken und sparen sparen sparen",
        "mega angebot heute -70% nur ueber meinen link!!!"]
ENGLISH = [("Sound quality is great but the app is a disaster.", "mixed"),
           ("Battery life is amazing, best earbuds I have owned.", "positive"),
           ("Stopped working after a week, very disappointed.", "negative"),
           ("Comfortable fit and clear sound, happy with the purchase.", "positive"),
           ("The noise cancelling is weak and the case feels cheap.", "negative")]
JUNK = ["", "   ", ".", "???", "kein kommentar", "-", "n/a", "...", "!!", "??", "keine angabe", "test"]


def _pos(rng):
    o = rng.choice(POS_OPENERS); a = rng.sample(POS, rng.choice([1, 2]))
    return f"{o} {' und '.join(a)}.", "positive"

def _neg(rng):
    o = rng.choice(NEG_OPENERS); a = rng.sample(NEG, rng.choice([1, 2]))
    return f"{o} {' und '.join(a)}.", "negative"

def _mixed(rng):
    p = rng.choice(POS); n = rng.choice(NEG)
    conn = rng.choice([" - aber ", ", allerdings ", ". Leider ", ", jedoch "])
    return f"{p[0].upper()+p[1:]}{conn}{n}.", "mixed"

def _neutral(rng):
    a = rng.choice(POS + NEG)
    t = rng.choice(NEU_TEMPLATES).format(a=a, a_cap=a[0].upper()+a[1:])
    return t, "neutral"

def _rating_for(truth, rng):
    return {"positive": [4,5,5], "negative": [1,1,2], "mixed": [2,3,4],
            "neutral": [3,3,4], "fake": [5,5], }.get(truth, [1,3,5]) and \
           rng.choice({"positive":[4,5,5],"negative":[1,1,2],"mixed":[2,3,4],
                       "neutral":[3,3,4],"fake":[5,5]}.get(truth,[1,3,5]))

def generate(n=300, seed=SEED):
    rng = random.Random(seed)
    start = date(2026, 1, 1)
    rows = []
    for i in range(n):
        r = rng.random()
        if r < 0.34:   text, truth = _pos(rng)
        elif r < 0.60: text, truth = _neg(rng)
        elif r < 0.72: text, truth = _mixed(rng)
        elif r < 0.80: text, truth = _neutral(rng)
        elif r < 0.88: text, truth = rng.choice(SARCASTIC), "negative"
        elif r < 0.93: text, truth = rng.choice(FAKE), "fake"
        elif r < 0.98: text, truth = rng.choice(ENGLISH)
        else:          text, truth = rng.choice(JUNK), "junk"
        day = int(rng.triangular(0, 270, 200 if truth == "negative" else 90))
        rows.append({
            "review_id": f"R{i:04d}",
            "date": (start + timedelta(days=day)).isoformat(),
            "product": PRODUCT,
            "rating": _rating_for(truth, rng),
            "text": text,
            "true_sentiment": truth,
            "is_sarcastic": text in SARCASTIC,
            "is_fake": text in FAKE,
        })
    # genau 3 absichtliche exakte Duplikate für die Dedup-Lektion
    if n > 20:
        for j, src in enumerate([5, 12, 30]):
            rows.append(dict(rows[src], review_id=f"R{n+j:04d}"))
    rng.shuffle(rows)
    return rows


def save(rows, csv_path, hide_truth=False):
    import csv
    fields = ["review_id","date","product","rating","text"]
    if not hide_truth: fields += ["true_sentiment","is_sarcastic","is_fake"]
    with open(csv_path,"w",newline="",encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader(); w.writerows(rows)

Die letzten drei Spalten sind die **Wahrheits-Marker**. Achtung: `true_sentiment` ist **unsere Festlegung** (erinnert euch an die Abstimmung in Sitzung 1). Sie enthält außerdem zwei Werte, die gar kein Sentiment sind (fake und junk). Für heute filtern wir `fake` und `junk` einfach heraus. Den sauberen Umgang mit solchen Daten machen wir in Sitzung 4 (Datenqualität) und Sitzung 10 (Fakes).

In [21]:
df["true_sentiment"].value_counts()

,count
true_sentiment,
negative,107
positive,107
mixed,44
neutral,25
fake,12
junk,8


In [5]:
df = pd.DataFrame(generate(300))
print(df.shape)
df.head(8)

(303, 8)


,review_id,date,product,rating,text,true_sentiment,is_sarcastic,is_fake
0,R0135,2026-05-24,Nimbus Q2,1,Ärgerlich: sie fallen leicht aus dem Ohr.,negative,False,False
1,R0001,2026-04-13,Nimbus Q2,4,Was soll ich sagen – die Passform ist perfekt.,positive,False,False
2,R0244,2026-04-15,Nimbus Q2,4,Volle Punktzahl: Bluetooth verbindet sofort und satte Bässe.,positive,False,False
3,R0297,2026-06-08,Nimbus Q2,3,"Habe sie seit Kurzem, die Geräuschunterdrückung rauscht – kann noch nicht viel sagen.",neutral,False,False
4,R0084,2026-01-25,Nimbus Q2,2,"Der Akku hält den ganzen Tag, allerdings viel zu teuer.",mixed,False,False
5,R0097,2026-07-06,Nimbus Q2,4,Sound quality is great but the app is a disaster.,mixed,False,False
6,R0045,2026-05-22,Nimbus Q2,3,!!,junk,False,False
7,R0265,2026-08-28,Nimbus Q2,1,Enttäuschend: die Touch-Steuerung reagiert kaum und die Geräuschunterdrückung rauscht.,negative,False,False


### ✏️ Eure Aufgabe: den Datensatz kennenlernen (10 min)

Beantwortet mit pandas:

1. Wie hoch ist der Anteil sarkastischer Bewertungen (`is_sarcastic`)?
2. Wie hoch ist die mittlere Sternebewertung je `true_sentiment`? Wo fällt euch etwas auf?
3. Wie viele **Duplikate** gibt es? Vorsicht, die Antwort hängt davon ab, was ihr als „doppelt" definiert.
4. Wie verteilt sich die Textlänge (Zeichen)? Gibt es Ausreißer nach unten?

In [19]:
# 1. Wie hoch ist der Anteil sarkastischer Bewertungen?
#    Tipp: is_sarcastic ist True/False. Was liefert .mean() auf so einer Spalte?
#    Achtet auf: Ist das viel oder wenig, wenn ihr an eure eigenen Bewertungen denkt?




# 2. Wie viele Sterne gibt es im Schnitt je true_sentiment?
#    Tipp: df.groupby(...)[...].mean()
#    Achtet auf: Welche Gruppe hat die meisten Sterne? Passt das zum Inhalt?



# 3. Wie viele Duplikate gibt es?
#    Tipp: .duplicated().sum(), einmal auf ganze Zeilen, einmal nur auf "text"
#    Achtet auf: Warum unterscheiden sich die beiden Zahlen so stark?
#    (review_id ist immer eindeutig, lasst sie beim Zeilenvergleich weg.)




# 4. Wie lang sind die Texte (in Zeichen)?
#    Tipp: df["text"].str.len(), dann .describe(); kürzeste Texte mit .nsmallest()
#    Achtet auf: Was steht in den allerkürzesten Zeilen?




# 5. Wie verteilen sich die Sterne (1–5)?
#    Tipp: df["rating"].value_counts().sort_index()
#    Achtet auf: J-Form wie bei echten Bewertungen, oder etwas anderes?



In [20]:
# Lösung: erst selbst versuchen, dann ausführen und vergleichen.
print("1. Anteil sarkastisch:", round(df["is_sarcastic"].mean(), 3))
print("\n2. Durchschnittliche Sterne je true_sentiment:")
print(df.groupby("true_sentiment")["rating"].mean().round(2).sort_values())
print("\n2b. Sarkastische Bewertungen: Wahrheit und Sterne")
print(df[df["is_sarcastic"]]["true_sentiment"].value_counts())
print("Durchschnittliche Sterne:", round(df[df["is_sarcastic"]]["rating"].mean(), 2))
print("Beispiel:", df[df["is_sarcastic"]]["text"].iloc[0])
print("\n3a. Komplett doppelte Zeilen (ohne review_id):",
      df.drop(columns="review_id").duplicated().sum())
print("3b. Doppelte Texte:", df["text"].duplicated().sum(),
      "-> v. a. Müll-Zeilen und wiederkehrende Sarkasmus-/Fake-Texte")
laenge = df["text"].str.len()
print("\n4. Textlänge:"); print(laenge.describe().round(1))
print("\nKürzeste Texte:", df.loc[laenge.nsmallest(5).index, "text"].tolist())

1. Anteil sarkastisch: 0.086

2. Durchschnittliche Sterne je true_sentiment:
true_sentiment
negative    1.36
mixed       2.98
neutral     3.36
junk        3.75
positive    4.69
fake        5.00
Name: rating, dtype: float64

2b. Sarkastische Bewertungen: Wahrheit und Sterne
true_sentiment
negative    26
Name: count, dtype: int64
Durchschnittliche Sterne: 1.27
Beispiel: Fantastisch leise – weil nach zwei Tagen einfach tot.

3a. Komplett doppelte Zeilen (ohne review_id): 4
3b. Doppelte Texte: 49 -> v. a. Müll-Zeilen und wiederkehrende Sarkasmus-/Fake-Texte

4. Textlänge:
count    303.0
mean      58.8
std       17.1
min        0.0
25%       49.0
50%       60.0
75%       70.5
max       94.0
Name: text, dtype: float64

Kürzeste Texte: ['', '!!', '!!', '??', '!!']


> 🎓 **Beobachtung:** Fakes haben die höchsten Sterne, aber keinen echten Inhalt. Sarkastische Bewertungen haben wenige Sterne und sind laut Wahrheit *negativ*, klingen aber positiv („Super, schon nach drei Tagen kaputt").
> Sterne, Wörter und Bedeutung erzählen nicht immer dieselbe Geschichte. Deshalb brauchen wir ein Modell, das **Bedeutung** versteht, nicht nur Wörter zählt.

## 2. Anatomie eines API-Aufrufs

Ein LLM-Aufruf ist ein HTTP-Request: ein JSON-Dokument hin, ein JSON-Dokument zurück. So sieht die Anfrage aus, die bei Anthropic ankommt:

In [28]:
bewertung = df.loc[0, "text"]

request = {
    "model": ANTHROPIC_MODEL,
    "max_tokens": 300,
    "system": "Du bist ein Analyst für Kundenfeedback.",              # Rolle & Regeln
    "messages": [{"role": "user",                                      # die eigentliche Aufgabe
                  "content": f'Wie ist das Sentiment dieser Bewertung?\n"""{bewertung}"""'}],
}
print(json.dumps(request, indent=2, ensure_ascii=False))

{
  "model": "claude-haiku-4-5",
  "max_tokens": 300,
  "system": "Du bist ein Analyst für Kundenfeedback.",
  "messages": [
    {
      "role": "user",
      "content": "Wie ist das Sentiment dieser Bewertung?\n\"\"\"Ärgerlich: sie fallen leicht aus dem Ohr.\"\"\""
    }
  ]
}


| Feld | Bedeutung |
|---|---|
| `model` | welches Modell; bestimmt Qualität, Preis und erlaubte Parameter |
| `max_tokens` | Obergrenze für die **Antwort**, schneidet hart ab |
| `system` | Rolle, Regeln, Format: gilt für das ganze Gespräch |
| `messages` | der Verlauf; hier eine einzige User-Nachricht |

Jetzt der Aufruf selbst. Schaut euch **alle** Felder der Antwort an, nicht nur den Text:

In [23]:
antwort = call_llm(request["messages"][0]["content"], system=request["system"])
for k, v in antwort.items():
    if k != "raw":
        print(f"{k:14}: {v}")

text          : # Sentiment-Analyse

**Sentiment: NEGATIV** 😞

## Begründung:

- **Ärgerlich** - Das Wort am Anfang signalisiert direkt eine negative Emotion
- **"fallen leicht aus dem Ohr"** - Dies beschreibt ein praktisches Problem/Mangel des Produkts
- **Impliziter Ärger** - Der Ton deutet auf Frustration hin, da das Produkt seinen Zweck (im Ohr bleiben) nicht erfüllt

## Zusammenfassung:
Die Bewertung drückt **Unzufriedenheit** aus. Der Nutzer kritisiert die schlechte Passform/Haftung des Produkts und empfindet das als ärgerlich.
input_tokens  : 51
output_tokens : 195
stop_reason   : end_turn
model         : claude-haiku-4-5-20251001


- `stop_reason`: Warum hat das Modell aufgehört? `end_turn` heißt „fertig", `max_tokens` heißt „abgeschnitten". Ein abgeschnittenes JSON ist kaputtes JSON.
- `input_tokens` / `output_tokens`: Danach wird abgerechnet. Ein Token ist ein Wortstück, grob 3–4 Zeichen im Deutschen. *(Im Mock nur geschätzt.)*

### ✏️ Kosten überschlagen (5 min)

Rechnet aus, was die Einordnung **aller 303** Bewertungen kostet, und was es bei **100.000** Bewertungen wären.

In [29]:
# Beispielpreise in USD pro 1 Mio. Tokens. Für euer Modell aktuelle Preise prüfen!
# (Werte hier: Claude Haiku 4.5 laut Anthropic-Doku, Stand 10/2026)
PREIS_INPUT_PRO_MTOK  = 1.00
PREIS_OUTPUT_PRO_MTOK = 5.00

kosten_pro_aufruf = ...   # TODO: aus antwort["input_tokens"] und antwort["output_tokens"]

In [30]:
# Lösung: erst selbst versuchen, dann ausführen und vergleichen.
kosten_pro_aufruf = (antwort["input_tokens"] * PREIS_INPUT_PRO_MTOK +
                     antwort["output_tokens"] * PREIS_OUTPUT_PRO_MTOK) / 1_000_000
for n in [1, len(df), 100_000]:
    print(f"{n:>7} Bewertungen: {n * kosten_pro_aufruf:10.4f} USD")

      1 Bewertungen:     0.0010 USD
    303 Bewertungen:     0.3109 USD
 100000 Bewertungen:   102.6000 USD


> 💡 Bei kleinen Mengen spielt Geld keine Rolle, bei Big Data schon. Was bei 100.000 oder Millionen Aufrufen sonst noch zählt (Latenz, Rate Limits, Batch-Rabatte), vertiefen wir in Sitzung 11.

## 3. Prompt als Funktionsdesign

Jetzt bauen wir das Herzstück des Review Radar: eine Funktion **`classify(text)`**, die eine Bewertung bekommt und ein Label zurückgibt, also `"positive"`, `"negative"`, `"mixed"` oder `"neutral"`.

Von außen sieht sie aus wie jede andere Python-Funktion. Innen ist sie etwas völlig anderes:

**Klassisch:** Die Logik steht als Code da.
```python
def classify(text):
    if "kaputt" in text or "enttäuscht" in text:
        return "negative"
    return "positive"
```

**Mit LLM:** Die Logik steht als *Anweisung in Sprache* da.
```python
def classify(text):
    antwort = call_llm(f'Ordne das Sentiment ein: """{text}"""', system=PROMPT)
    return antwort["text"]      # Freitext, muss erst geprüft werden
```

| | klassische Funktion | LLM-Funktion |
|---|---|---|
| Logik | Code, Regel für Regel | ein **Prompt** in natürlicher Sprache |
| Gleiche Eingabe | immer gleiches Ergebnis | **kann variieren** (Block 3) |
| Rückgabe | genau der versprochene Typ | **Freitext**, muss erst geparst werden |
| Fehler | Absturz mit Fehlermeldung | **plausibel falsch**, ohne Warnung |
| Kosten | praktisch null | Geld + ca. 1 Sekunde pro Aufruf |
| Stärke | exakt, aber starr: „super, kaputt" überfordert sie | versteht Bedeutung, auch Sarkasmus |

Deshalb gilt: Bevor wir den Prompt schreiben, brauchen wir eine **Spezifikation**: Welche Labels gibt es, und wann gilt welches?

| Label | Definition (unsere Festlegung) |
|---|---|
| `positive` | überwiegend Lob, keine nennenswerte Kritik |
| `negative` | überwiegend Kritik; **auch** wenn positiv formuliert (Sarkasmus) |
| `mixed` | konkretes Lob **und** konkrete Kritik zu verschiedenen Aspekten |
| `neutral` | weder klar positiv noch negativ; reine Feststellung |
> 🎓 Erinnert euch an die Abstimmung aus Sitzung 1: Ohne diese Tabelle gibt es kein „richtig" oder „falsch". Die Tabelle **ist** die Wahrheit, gegen die wir später messen.

> ⚠️ `fake` und `junk` kommen in unserer Spezifikation bewusst nicht vor: Sie sind keine Stimmungen. Beim Abgleich mit der Wahrheit (Abschnitt 4) lassen wir sie weg. Läuft der Klassifikator aber über *alle* Daten, muss er sie in eines der vier Labels pressen, und ein Fake wird dann wohl als `positive` gezählt. Wie man solche Zeilen vorher aussortiert, ist Thema in Sitzung 4 und 10.

### ✏️ Eure Aufgabe: `classify()` bauen (20 min)

Drei Teile:

1. **`SYSTEM_PROMPT`**: Schreibt die Regeln. Nutzt die Label-Tabelle oben und verlangt **nur JSON** als Antwort: `{"sentiment": "...", "reason": "..."}`.
2. **`parse_label(raw)`**: Verwandelt die Rohantwort in ein gültiges Label. Robust gegen: Text vor/nach dem JSON, ```` ```json ````-Blöcke, ungültige Labels. Bei Fehlern `"invalid"` zurückgeben, **nicht** abstürzen.
3. **`classify(text)`**: Setzt beides zusammen.

Tipp: Die Bewertung immer zwischen `"""` setzen. Das trennt **Daten** von **Anweisungen**. (Der Mock braucht das auch, um die Bewertung zu finden.)

In [31]:
LABELS = ["positive", "negative", "mixed", "neutral"]

SYSTEM_PROMPT = """
TODO: Rolle, Label-Definitionen, Ausgabeformat
"""

def parse_label(raw):
    # TODO: JSON aus raw herausziehen, "sentiment" lesen, gegen LABELS prüfen
    raise NotImplementedError

def classify(text, system=SYSTEM_PROMPT, temperature=None):
    user = f'Bewertung:\n"""{text}"""'
    raw = call_llm(user, system=system, temperature=temperature)["text"]
    return parse_label(raw)

In [32]:
# Lösung: erst selbst versuchen. Ausführen, falls ihr nicht fertig werdet
# (die folgenden Zellen brauchen ein funktionierendes classify()).
LABELS = ["positive", "negative", "mixed", "neutral"]

SYSTEM_PROMPT = """Du klassifizierst Produktbewertungen nach Sentiment.

Labels:
- positive: überwiegend Lob, keine nennenswerte Kritik
- negative: überwiegend Kritik, auch wenn sie positiv formuliert ist (Sarkasmus)
- mixed: konkretes Lob UND konkrete Kritik zu verschiedenen Aspekten
- neutral: weder klar positiv noch negativ, reine Feststellung

Antworte ausschließlich mit JSON, ohne weiteren Text:
{"sentiment": "<label>", "reason": "<max. 15 Wörter>"}"""

def parse_json(raw):
    """Zieht das JSON-Objekt aus der Rohantwort. None, wenn das nicht klappt."""
    m = re.search(r"\{.*\}", raw or "", re.DOTALL)   # vom ersten { bis zum letzten }
    if not m:
        return None
    try:
        daten = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    return daten if isinstance(daten, dict) else None   # z. B. Liste statt Objekt

def parse_label(raw):
    daten = parse_json(raw)
    if daten is None:
        return "invalid"
    label = str(daten.get("sentiment", "")).strip().lower()
    return label if label in LABELS else "invalid"

def classify(text, system=SYSTEM_PROMPT, temperature=None):
    user = f'Bewertung:\n"""{text}"""'
    raw = call_llm(user, system=system, temperature=temperature)["text"]
    return parse_label(raw)

def classify_mit_grund(text, system=SYSTEM_PROMPT):
    """Wie classify, gibt aber Label UND Begründung zurück (für die Fehleranalyse)."""
    user = f'Bewertung:\n"""{text}"""'
    raw = call_llm(user, system=system)["text"]
    daten = parse_json(raw) or {}
    return {"label": parse_label(raw), "reason": daten.get("reason", raw[:80])}

Testet `parse_label` an den Fällen, die echte Modelle tatsächlich (leider) liefern:

In [33]:
tests = {
    # Idealfall: sauberes JSON, wie im Prompt verlangt
    '{"sentiment": "positive", "reason": "Lob"}': "positive",

    # Geschwätzig + Markdown: Text davor, JSON im Code-Block, Label großgeschrieben
    'Klar! Hier: ```json\n{"sentiment": "Negative"}\n```': "negative",

    # Gültiges JSON, aber Label außerhalb unserer Spezifikation
    '{"sentiment": "sehr gut"}': "invalid",

    # Gar kein JSON: Modell ignoriert die Formatvorgabe
    'Die Bewertung ist positiv.': "invalid",

    # Abgeschnitten: max_tokens erreicht, JSON unvollständig
    '{"sentiment": "mixed", "reason": "abgeschnitten': "invalid",

    # Liste statt Objekt: Parser greift das Objekt in der Liste heraus (gewollt)
    '[{"sentiment": "positive"}]': "positive",

    # Zwei JSON-Objekte: Modell wiederholt erst das Format, dann die Antwort
    'Format: {"sentiment": "<label>"}\nAntwort: {"sentiment": "negative"}': "negative",
}
for raw, erwartet in tests.items():
    got = parse_label(raw)
    print("✅" if got == erwartet else "❌", repr(raw[:45]), "->", got)

✅ '{"sentiment": "positive", "reason": "Lob"}' -> positive
✅ 'Klar! Hier: ```json\n{"sentiment": "Negative"}' -> negative
✅ '{"sentiment": "sehr gut"}' -> invalid
✅ 'Die Bewertung ist positiv.' -> invalid
✅ '{"sentiment": "mixed", "reason": "abgeschnitt' -> invalid
✅ '[{"sentiment": "positive"}]' -> positive
❌ 'Format: {"sentiment": "<label>"}\nAntwort: {"s' -> invalid


> 🎓 **Ein Test schlägt fehl.** Warum? Und die eigentliche Frage: Sollten wir den Parser „reparieren", oder ist `invalid` hier sogar die *sicherere* Antwort?
> Ein Parser, der rät, liefert manchmal ein falsches Label, ohne dass es jemand merkt. Ein Parser, der `invalid` sagt, ist ehrlich, verliert aber Daten. Robuste Lösungen (Schema-Validierung, automatische Wiederholung) bauen wir in Sitzung 6.

In [34]:
classify("Der Klang ist hervorragend, aber die App stürzt ständig ab.")

'mixed'

## 4. Erster Abgleich mit der Wahrheit

Wir nehmen eine **feste Stichprobe** von 20 Bewertungen mit echtem Sentiment-Label (Fakes und Junk lassen wir heute weg) und vergleichen.

> Das ist bewusst klein: Das Einordnen in großen Mengen ist Thema in Sitzung 3, sauberes Messen in Sitzung 12.

In [ ]:
probe = (df[df["true_sentiment"].isin(LABELS)]
         .sample(20, random_state=7)
         .reset_index(drop=True))

probe["pred_zero"] = [classify(t) for t in probe["text"]]
treffer = (probe["pred_zero"] == probe["true_sentiment"]).mean()
print(f"Accuracy zero-shot: {treffer:.0%}")
pd.crosstab(probe["true_sentiment"], probe["pred_zero"], rownames=["wahr"], colnames=["vorhergesagt"])

### Wozu eine Zahl allein nicht reicht

Eine Accuracy von „93 %" klingt nach einer klaren Aussage. Für Entscheidungen braucht man aber zusätzlich: **Wie sicher ist diese Zahl?** Also einen Bereich, in dem der wahre Wert plausibel liegt. Fünf Situationen, in denen genau das den Unterschied macht:

**1. Go / No-Go gegen eine Anforderung**
Vorgabe des Fachbereichs: „mindestens 90 % richtig". Entscheidend ist die **Untergrenze**, nicht der Mittelwert:

| Messung | Plausibler Bereich | Entscheidung |
|---|---|---|
| 94 % | 92–96 % | ✅ Go: auch im ungünstigsten Fall über 90 % |
| 93 % | 85–97 % | ⚠️ Nicht belegbar: mehr Daten prüfen |
| 84 % | 80–88 % | ❌ No-Go: auch im besten Fall unter 90 % |

Zeile 1 und 2 haben fast denselben Wert und führen trotzdem zu gegensätzlichen Entscheidungen.

**2. Alternativen vergleichen** (Prompt A vs. B, Modell X vs. Y)
Überlappen sich die Bereiche stark, gibt es keinen belegten Unterschied. Dann gewinnt die günstigere Variante.

**3. In Geschäftszahlen übersetzen**
10.000 Bewertungen pro Monat bei 92–96 % Accuracy = **400 bis 800 Fehlzuordnungen pro Monat**. Wie viel menschliche Nachkontrolle braucht das? Und wie stark kann der „Anteil negativer Bewertungen" im Dashboard danebenliegen?

**4. Das Labeling-Budget planen**
Wie viele Bewertungen müssen von Hand geprüft werden, um auf ±2 Prozentpunkte genau zu sein? Bei rund 93 % Accuracy: etwa **625**. Damit hat „Vertrauen" einen Preis.

**5. Überwachen über die Zeit**
Jeden Monat eine kleine Stichprobe neu prüfen. Liegt der neue Wert außerhalb des alten Bereichs, hat sich etwas verändert: neues Produkt, neue Kundschaft, neues Modell (*Drift*).

> 🎓 **Für eure Prüfung:** Nicht „unser Tool ist zu 93 % genau", sondern „93 % (95 %-Konfidenzintervall: 90–95 %, n = 500 geprüfte Bewertungen), das reicht für die Anforderung ≥ 85 %, unsichere Fälle prüft ein Mensch". **Genau das heißt „Vertrauen verdient".**

### Wie belastbar ist diese Zahl?

19 von 20 richtig klingt nach „95 % Accuracy". Aber das ist eine **Stichprobe**. Uns interessiert die Accuracy auf *allen* Bewertungen, und die kennen wir nicht.

Ein **95 %-Konfidenzintervall** gibt den Bereich an, in dem der wahre Wert plausibel liegt. Genauer: Würden wir immer wieder 20 Bewertungen ziehen und jedes Mal so ein Intervall berechnen, läge der wahre Wert in rund 95 % dieser Intervalle.

Wir nutzen das **Wilson-Intervall**. Die Lehrbuchformel (p ± 1,96·√(p(1−p)/n)) versagt bei kleinem n und Werten nahe 100 %: Für 19/20 liefert sie 85 % bis *105 %*, für 20/20 sogar „keinerlei Unsicherheit". Wilson bleibt zwischen 0 und 100 % und ist für Anteile der Standard.

In [27]:
def wilson(k, n, z=1.96):
    p = k / n
    mitte = (p + z**2 / (2*n)) / (1 + z**2 / n)
    breite = z * math.sqrt(p*(1-p)/n + z**2/(4*n**2)) / (1 + z**2 / n)
    return mitte - breite, mitte + breite

k = int((probe["pred_zero"] == probe["true_sentiment"]).sum())
lo, hi = wilson(k, len(probe))
print(f"{k}/{len(probe)} richtig -> 95%-KI: {lo:.0%} bis {hi:.0%}")

NameError: name 'probe' is not defined

### ✋ Experiment: 20 Stichproben, 20 Intervalle

Jede:r zieht eine **eigene** Stichprobe von 20 Bewertungen, klassifiziert sie und berechnet das 95 %-Intervall. Danach sammeln wir alle Ergebnisse und prüfen: Wie viele Intervalle enthalten die Accuracy auf *allen* Daten?

Wenn die Theorie stimmt: etwa 19 von 20.

In [ ]:
# Eure eigene Stichprobe: wählt eine beliebige Zahl (z. B. euren Geburtstag als 1403)
MEINE_ZAHL = 1403

meine_probe = df[df["true_sentiment"].isin(LABELS)].sample(20, random_state=MEINE_ZAHL)
meine_preds = [classify(t) for t in meine_probe["text"]]
k_meins = int((pd.Series(meine_preds, index=meine_probe.index) == meine_probe["true_sentiment"]).sum())
lo, hi = wilson(k_meins, 20)
print(f"Meldet: {k_meins}/20 richtig  ->  95%-KI: {lo:.0%} bis {hi:.0%}")

In [ ]:
# Nur Vortragende:r: Accuracy auf ALLEN Sentiment-Bewertungen (ca. 283 Aufrufe)
alle = df[df["true_sentiment"].isin(LABELS)]
alle_preds = [classify(t) for t in alle["text"]]
wahrer_wert = (pd.Series(alle_preds, index=alle.index) == alle["true_sentiment"]).mean()
print(f"Accuracy auf allen {len(alle)} Bewertungen: {wahrer_wert:.1%}")

In [ ]:
# Die gemeldeten Ergebnisse (Treffer von 20) hier eintragen:
ergebnisse = [19, 18, 17, 19, 20, 16, 18, 19, 17, 18, 15, 19, 18, 20, 17, 18, 19, 16, 18, 19]

fig, ax = plt.subplots(figsize=(8, 0.35 * len(ergebnisse) + 1))
treffer_zahl = 0
for i, k in enumerate(ergebnisse):
    lo, hi = wilson(k, 20)
    drin = lo <= wahrer_wert <= hi
    treffer_zahl += drin
    farbe = "#4f46e5" if drin else "#dc2626"
    ax.plot([lo, hi], [i, i], color=farbe, lw=3, solid_capstyle="round")
    ax.plot(k / 20, i, "o", color=farbe, ms=6)
ax.axvline(wahrer_wert, color="#111827", ls="--", lw=1.5, label=f"Wert auf allen Daten: {wahrer_wert:.0%}")
ax.set_xlim(0.4, 1.02)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_yticks(range(len(ergebnisse)))
ax.set_yticklabels([f"Probe {i+1}" for i in range(len(ergebnisse))], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Accuracy (Punkt = Messung, Linie = 95%-Konfidenzintervall)")
ax.legend(loc="lower left", fontsize=8, frameon=False)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()
print(f"{treffer_zahl} von {len(ergebnisse)} Intervallen enthalten den Wert auf allen Daten "
      f"({treffer_zahl/len(ergebnisse):.0%}; erwartet: ca. 95 %).")

> 🎓 **Was sehen wir?**
> - Jede Stichprobe liefert eine andere Accuracy (Punkte), keine ist „die richtige".
> - Die meisten Intervalle überdecken den Wert auf allen Daten, einzelne nicht (rot). Das ist kein Fehler, sondern genau, was „95 %" bedeutet.
> - Niemand wusste vorher, ob sein eigenes Intervall rot sein würde.

> 💡 Bei uns kennen wir die Wahrheit für alle Zeilen, weil wir die Daten selbst gebaut haben. In der Praxis muss jemand sie **von Hand labeln**, und das macht niemand für 10 Mio. Bewertungen. Man labelt eine Stichprobe und schließt mit einem Konfidenzintervall auf den Rest. Genau das machen wir in Sitzung 12.

Welche Fälle gehen schief?

In [ ]:
probe.loc[probe["pred_zero"] != probe["true_sentiment"],
          ["text", "true_sentiment", "pred_zero", "is_sarcastic"]]

In [ ]:
fehler = probe[probe["pred_zero"] != probe["true_sentiment"]]
print(f"{len(fehler)} Fehler von {len(probe)} im ersten Durchlauf\n")
for _, zeile in fehler.iterrows():
    erg = classify_mit_grund(zeile["text"])
    geaendert = "  ⚠️ andere Antwort als beim ersten Mal!" if erg["label"] != zeile["pred_zero"] else ""
    print(f'„{zeile["text"]}"')
    print(f'   wahr: {zeile["true_sentiment"]:9} 1. Lauf: {zeile["pred_zero"]:9} '
          f'2. Lauf: {erg["label"]:9}{geaendert}')
    print(f'   Begründung (2. Lauf): {erg["reason"]}\n')

### ✏️ Few-shot: Beispiele in den Prompt (10 min)

Statt nur Regeln zu beschreiben, kann man dem Modell **Beispiele** zeigen (*few-shot*). Ergänzt `FEW_SHOT` um 3–4 Beispiele, gerne einen sarkastischen Fall, und vergleicht die Accuracy.

> ⚠️ Nehmt Beispiele **nicht** aus der Stichprobe `probe`. Sonst prüft ihr das Modell an Aufgaben, deren Lösung im Prompt steht (*Data Leakage*).

In [ ]:
FEW_SHOT = """
Beispiele:
Bewertung: \"\"\"Toll, schon wieder kaputt. Super Qualität.\"\"\"
{"sentiment": "negative", "reason": "Sarkasmus: Lobwörter, aber Defekt"}
"""
# TODO: weitere Beispiele ergänzen

probe["pred_few"] = [classify(t, system=SYSTEM_PROMPT + "\n" + FEW_SHOT) for t in probe["text"]]
for spalte in ["pred_zero", "pred_few"]:
    print(f"{spalte}: {(probe[spalte] == probe['true_sentiment']).mean():.0%}")

> 🎓 **Mit dem Mock** ist jeder Unterschied hier **reiner Zufall**: Er ignoriert den Prompt und würfelt bei jedem Lauf neu. Führt die Zelle zweimal aus und vergleicht. Mit einem echten Modell wirkt der Prompt tatsächlich. Genau das ist der Unterschied zwischen Schlüsselwort-Zählen und Sprachverständnis.
>
> 🎓 **Und:** Ist ein Unterschied von 2–3 Treffern bei n = 20 überhaupt ein Unterschied? Schaut auf die Konfidenzintervalle.

## 5. Warum dieselbe Frage verschiedene Antworten gibt

Ein LLM erzeugt Text **Token für Token**. Bei jedem Schritt berechnet es für jedes mögliche nächste Token einen Score (*Logit*) und macht daraus Wahrscheinlichkeiten, mit der **Softmax mit Temperatur** $T$:

$$P(i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

Dann wird **gezogen**, nicht das Maximum genommen. Kleines $T$ schärft die Verteilung (fast immer das wahrscheinlichste Token), großes $T$ flacht sie ab (mehr Vielfalt, mehr Ausreißer). $T \to 0$ heißt: immer das Maximum (*greedy*).

Stellt euch vor, das Modell steht beim Antworten genau an der Stelle `{"sentiment": "` und hat diese Logits für die vier Labels:

In [ ]:
labels = ["positive", "negative", "mixed", "neutral"]
logits = np.array([2.1, 0.3, 1.6, -0.5])     # Spielzahlen

def softmax(z, T):
    z = np.asarray(z, dtype=float) / T
    e = np.exp(z - z.max())
    return e / e.sum()

temps = [0.2, 0.7, 1.0, 1.5]
fig, axes = plt.subplots(1, len(temps), figsize=(13, 2.8), sharey=True)
for ax, T in zip(axes, temps):
    p = softmax(logits, T)
    ax.bar(labels, p, color="#4f46e5")
    ax.set_title(f"T = {T}")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=45)
    for i, v in enumerate(p):
        ax.text(i, v + 0.03, f"{v:.0%}", ha="center", fontsize=8)
axes[0].set_ylabel("Wahrscheinlichkeit")
plt.tight_layout(); plt.show()

## 6. Experiment: die Streuung messen

Jetzt mit unserem Klassifikator: **dieselbe** Bewertung 10× einordnen, bei zwei Temperaturen. Wir messen

- die **Label-Verteilung** über die 10 Läufe,
- die **Übereinstimmungsrate**: Anteil der Läufe, die das häufigste Label liefern (1.0 = immer gleich).

Wir nehmen drei Bewertungen: eine eindeutige, eine gemischte, eine sarkastische.

In [ ]:
# Wo ist das Modell unsicher? Jede Bewertung der Stichprobe 5x einordnen
scan = []
for t in probe["text"]:
    c = Counter(classify(t, temperature=1.0) for _ in range(5))
    scan.append({"text": t, "verteilung": dict(c), "uebereinstimmung": c.most_common(1)[0][1] / 5})
scan = pd.DataFrame(scan).sort_values("uebereinstimmung")
scan.head(6)

In [ ]:
# Ein eindeutiger Fall als Kontrolle + die drei unsichersten Fälle aus dem Scan
faelle = {"eindeutig (Kontrolle)": df[df["true_sentiment"] == "positive"]["text"].iloc[0]}
for i, t in enumerate(scan["text"].head(3), 1):
    faelle[f"grenzfall {i}"] = t
for k, v in faelle.items():
    print(f"{k:22}: {v}")

In [ ]:
N = 10
zeilen = []
for name, text in faelle.items():
    for T in [0.0, 1.0]:
        try:
            labels_n = [classify(text, temperature=T) for _ in range(N)]
        except Exception as e:
            print(f"⚠️ {name}, T={T}: {type(e).__name__}: {str(e)[:120]}")
            continue
        c = Counter(labels_n)
        zeilen.append({"fall": name, "T": T,
                       "verteilung": dict(c),
                       "übereinstimmung": c.most_common(1)[0][1] / N})
ergebnis = pd.DataFrame(zeilen)
ergebnis

> ⚠️ **Gut zu wissen:** Nicht jedes Modell lässt euch die Temperatur wählen. Unser Haiku 4.5 erlaubt es; neuere Modelle wie Claude Sonnet 5.5 lehnen eine eigene Temperatur mit einer Fehlermeldung ab und sampeln immer mit ihrer Standardeinstellung. Das ist eine Entscheidung des Anbieters, kein Fehler in eurem Code. Offene Modelle (z. B. Llama oder Qwen) erlauben die Temperatur meist.

**Was ihr seht:**

- **T = 0:** Jede Bewertung liefert 10 von 10 Mal dasselbe Label. Das Modell nimmt immer die wahrscheinlichste Antwort.
- **T = 1:** Einige Bewertungen streuen, andere nicht. Wo das Modell zwischen zwei Labels schwankt, ist es auch „innerlich" unsicher.
- **„Eindeutig" ist relativ:** Was uns klar erscheint, muss für das Modell nicht klar sein. Auch unsere Kontrolle kann wackeln.
- **Vorsicht bei der Auswahl:** Manche Fälle, die im Scan (5 Läufe) unsicher wirkten, sind mit 10 Läufen stabil. Wer auf Basis weniger Messungen auswählt, erwischt auch Zufallstreffer. Das ist **Regression zur Mitte**.
- **Stabil heißt nicht richtig:** Ein Fall mit 10/10 gleichen Antworten kann trotzdem falsch sein, wenn das Modell sich konsequent irrt.

> 💡 **Kernpunkt:** Ein LLM-Klassifikator liefert keine feste Zahl, sondern eine **Stichprobe aus einer Verteilung**. Die Unsicherheit konzentriert sich auf Grenzfälle. Wer die Qualität messen will, braucht feste Stichproben, wiederholte Läufe und Intervalle statt Punktwerte. Und für unsichere Fälle lohnt sich ein Mensch im Prozess (Sitzung 13).

## Zum Weiterprobieren (freiwillig)

1. **Sarkasmus knacken:** Verbessert `SYSTEM_PROMPT` oder `FEW_SHOT`, bis möglichst viele der `is_sarcastic`-Bewertungen als `negative` erkannt werden. Belegt die Verbesserung mit Zahlen und einem Konfidenzintervall, nicht mit einem Einzelbeispiel.
2. **Temperatur-Kurve:** Variiert $T$ von 0 bis 2 und plottet die Übereinstimmungsrate für den gemischten Fall.
3. **Modellvergleich:** Wenn ihr Zugang zu zwei Backends habt: Welches ist bei Grenzfällen stabiler? Welches günstiger?
4. **Kosten:** Wie verändert ein längerer System-Prompt (z. B. mit 10 Few-shot-Beispielen) die Kosten pro Aufruf?

## Ausblick

**Nächste Woche (21.10.): Sentiment im Großen.** Von 20 auf alle Bewertungen: Batching, Fehlerbehandlung, die ersten aggregierten Kennzahlen, und die Frage, ob man überhaupt alle Daten braucht.